# WP4 / C3 - evaluation budget (B1): attempted vs unique evaluations, easy-article recovery, comparator settings

Runs on its own, Runtime -> Run all.

- reads from Drive: mowoa_mo_study/article_caches_150.pkl, phaseC/*, wp2/wp2_runs.csv
- only writes to mowoa_mo_study/wp4/ (re-running skips the articles already done)
- about 2-3 h on a Colab CPU (13,500 comparator reruns plus ~10 min of timing). The pre-check in C5 stops within a minute if this pymoo version doesn't reproduce phase C
- the files I keep afterwards: wp4/wp4_summary.txt and wp4/wp4_per_algorithm.csv
- article numbers in the CSVs are 1-based (Artikel 149 = 149), same as WP2

## C0 - setup

In [ ]:
!pip install pymoo==0.6.2 -q        # the version Phase C used (handoff §9.3); C5 checks reproduction first
import os, io, pickle, time, platform, inspect, contextlib
import numpy as np, pandas as pd
from scipy import stats
from multiprocessing import get_context
from concurrent.futures import ProcessPoolExecutor, as_completed
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/mowoa_mo_study'
PHASEC_DIR = f'{DRIVE_DIR}/phaseC'
WP2_RUNS   = f'{DRIVE_DIR}/wp2/wp2_runs.csv'
WP4_DIR    = f'{DRIVE_DIR}/wp4'
RUN_DIR    = f'{WP4_DIR}/peer_runs'
for d in (WP4_DIR, RUN_DIR):
    os.makedirs(d, exist_ok=True)
CACHE_PATH = f'{DRIVE_DIR}/article_caches_150.pkl'
SEEDS   = list(range(30))      # Phase C seeds
WORKERS = os.cpu_count() or 1  # articles in parallel (fork), same results per seed either way
RUN_TIMING   = True            # C8: sequential timing on the 33 hard articles (R1.3 data, up to the supervisor whether to report it)
TIMING_SEEDS = [0, 1, 2]


## C1 - shared module (same code as phase B/C/D)

In [ ]:
%%writefile mowoa_mo_shared.py
"""
shared layer for the MO-WOA multi-objective comparison (phase B, axis 2).

all four algorithms read the same enumerated objective table and use the same
encoding, repair, initial population per (article, seed) and FE budget, so
objective identity holds by construction.

protocol decisions: D-1 shared P0 (paired design), D-2 terminal + cumulative
fronts, D-3 B = 1480 = 30 + 50 x 29 (surplus goes to the peers, never MO-WOA),
D-4 no tuning, D-6 strata by m = |P*|: A m>=3, B m==2, C m==1 (degenerate).
summary_size=3, repair_solution(), position bonus, knee_weights=[32, 1], the
70/30 leader split and leader preservation are copied verbatim from cell 20 of [v1_4_1].
"""

import numpy as np
from itertools import combinations

# protocol constants (decisions D-1 to D-6 above), don't change them
K_SUMMARY      = 3                       # summary_size (locked)
POP_SIZE       = 30                      # MO-WOA locked
MAX_ITER       = 50                      # MO-WOA locked
FE_BUDGET      = 1480                    # 30 + 50*29   (D-3)
KNEE_WEIGHTS   = np.array([32.0, 1.0])   # w_cov = 32 (locked)
REDUNDANCY_OFFSET = 0.2                  # delta in f2
REF_POINT      = np.array([1.1, 1.1])    # HV reference, normalized space


# per-article artefacts, computed once (BGE-M3 is slow)
class ArticleCache:
    """Everything the optimizer needs for one article. No embeddings at run time."""

    def __init__(self, article_id, adj_matrix, node_importance, position_bonus):
        self.article_id        = article_id
        self.adj               = np.asarray(adj_matrix, dtype=float)
        self.node_importance   = np.asarray(node_importance, dtype=float)
        self.position_bonus    = np.asarray(position_bonus, dtype=float)
        self.n                 = len(self.node_importance)
        # same as cell 20: graph.nodes[i]['importance'] * position_bonus[i]
        self.biased_importance = self.node_importance * self.position_bonus

    def to_dict(self):
        return dict(article_id=self.article_id, adj=self.adj,
                    node_importance=self.node_importance,
                    position_bonus=self.position_bonus)

    @staticmethod
    def from_dict(d):
        return ArticleCache(d['article_id'], d['adj'],
                            d['node_importance'], d['position_bonus'])


def build_position_bonus(n):
    """Copied from cell 28 of [v1_4_1], don't touch."""
    pb = np.ones(n)
    pb[0]   = 3.0
    pb[1:3] *= 2.5
    pb[3:5] *= 1.5
    pb[5:8] *= 0.8
    pb[8:]  *= 0.3
    return pb


# objectives, one implementation for all four algorithms
def evaluate_objectives_reference(solution, cache):
    """
    Literal copy of cell 20's evaluate_objectives(), only kept as ground truth for
    G4 (slow double loop). Returns native (coverage max, redundancy min).
    """
    solution = np.asarray(solution).astype(int)
    selected = np.where(solution == 1)[0]
    coverage = cache.biased_importance[selected].sum() if len(selected) > 0 else 0
    redundancy_raw = 0.0
    num_pairs = 0
    for i_idx, i in enumerate(selected):
        for j_idx, j in enumerate(selected):
            if j_idx > i_idx:
                redundancy_raw += cache.adj[i][j]
                num_pairs += 1
    redundancy = redundancy_raw / max(1, num_pairs) if num_pairs > 0 else 0
    redundancy = max(0, redundancy - REDUNDANCY_OFFSET)
    return coverage, redundancy


def evaluate_objectives_vec(combos, cache):
    """Vectorized: (M, k) sorted combos -> (M, 2) native (coverage, redundancy). Checked by G4."""
    combos = np.asarray(combos, dtype=int)
    M, k = combos.shape
    coverage = cache.biased_importance[combos].sum(axis=1)

    n_pairs = k * (k - 1) // 2
    if n_pairs == 0:
        redundancy = np.zeros(M)
    else:
        iu, ju = np.triu_indices(k, k=1)          # all i<j pairs within a subset
        rows = combos[:, iu]                      # (M, n_pairs)
        cols = combos[:, ju]
        redundancy_raw = cache.adj[rows, cols].sum(axis=1)
        redundancy = redundancy_raw / n_pairs
        redundancy = np.maximum(0.0, redundancy - REDUNDANCY_OFFSET)

    return np.column_stack([coverage, redundancy])


# repair operator, copied from cell 20. don't touch
def repair_solution(solution, cache, target_length=K_SUMMARY):
    solution = np.asarray(solution).astype(int).copy()
    bi = cache.biased_importance
    selected = np.where(solution == 1)[0]
    num_selected = len(selected)

    if num_selected > target_length:
        drop_order = selected[np.argsort(bi[selected])]        # lowest importance first
        for idx in drop_order[:num_selected - target_length]:
            solution[idx] = 0
    elif num_selected < target_length:
        unselected = np.where(solution == 0)[0]
        if len(unselected) > 0:
            add_order = unselected[np.argsort(-bi[unselected])]  # highest importance first
            for idx in add_order[:target_length - num_selected]:
                solution[idx] = 1

    if not solution.any():
        solution[np.argmax(bi)] = 1
    return solution


class Enumeration:
    """All C(N, k) subsets of one article: the exact true front (not a pooled one) + O(1) lookups."""

    def __init__(self, cache, k=K_SUMMARY):
        self.cache = cache
        self.k = k
        n = cache.n
        if n < k:
            raise ValueError(f"article {cache.article_id}: N={n} < k={k}")

        self.combos = np.array(list(combinations(range(n), k)), dtype=np.int16)
        self.M = len(self.combos)
        self.F_native = evaluate_objectives_vec(self.combos, cache)   # (M,2) (cov, red)

        # minimization form used by pymoo and every indicator: (-coverage, redundancy)
        self.F_min = np.column_stack([-self.F_native[:, 0], self.F_native[:, 1]])

        # O(1) lookup:  sorted index tuple -> row in F_native
        self.index_of = {tuple(c): i for i, c in enumerate(map(tuple, self.combos))}

        # normalize on the full feasible set, never on algorithm output
        cov = self.F_native[:, 0]
        red = self.F_native[:, 1]
        self.cov_min, self.cov_max = cov.min(), cov.max()
        self.red_min, self.red_max = red.min(), red.max()
        self.cov_range = self.cov_max - self.cov_min
        self.red_range = self.red_max - self.red_min

        # exact true front, deduped in objective space
        nd = non_dominated_mask(self.F_min)
        F_pf = self.F_min[nd]
        self.P_star_min = unique_rows(F_pf)                 # (m, 2) minimization
        self.P_star_norm = self.normalize(self.P_star_min)
        self.m = len(self.P_star_min)                       # |P*|  -> stratum

        # HV of the true front = max attainable HV
        self.hv_star = hypervolume(self.P_star_norm, REF_POINT)

        # true coverage extreme + true knee, for the hit-rate metrics
        self.true_cov_extreme_idx = int(np.argmax(cov))
        self.true_knee_row = self._true_knee()

    def normalize(self, F_min):
        """Map minimization objectives into per-article [0,1]^2 (both minimized)."""
        F_min = np.atleast_2d(np.asarray(F_min, dtype=float))
        cov = -F_min[:, 0]
        red = F_min[:, 1]
        cr = self.cov_range if self.cov_range > 0 else 1.0
        rr = self.red_range if self.red_range > 0 else 1.0
        g1 = (self.cov_max - cov) / cr          # 0 = best coverage
        g2 = (red - self.red_min) / rr          # 0 = least redundant
        return np.column_stack([g1, g2])

    def F_of(self, solution):
        """O(1) lookup for a binary vector, native (coverage, redundancy)."""
        key = tuple(np.where(np.asarray(solution).astype(int) == 1)[0].tolist())
        return self.F_native[self.index_of[key]]

    def row_of(self, solution):
        key = tuple(np.where(np.asarray(solution).astype(int) == 1)[0].tolist())
        return self.index_of[key]

    def _true_knee(self):
        """Knee of the true front with the locked knee_weights = [32, 1]."""
        P = self.P_star_min
        if len(P) == 1:
            return 0
        native = np.column_stack([-P[:, 0], P[:, 1]])
        omin, omax = native.min(axis=0), native.max(axis=0)
        rng = omax - omin
        rng[rng == 0] = 1
        norm = (native - omin) / rng
        norm[:, 0] = 1 - norm[:, 0]                       # invert coverage
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        return int(d.argmin())

    @property
    def stratum(self):
        if self.m >= 3:
            return 'A'
        elif self.m == 2:
            return 'B'
        return 'C'


# dominance utils, minimization everywhere
def non_dominated_mask(F):
    """Mask of non-dominated rows, F is (n, 2) minimization."""
    F = np.atleast_2d(np.asarray(F, dtype=float))
    n = len(F)
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]:
            continue
        # j dominates i  iff  all(F[j] <= F[i]) and any(F[j] < F[i])
        le = np.all(F <= F[i], axis=1)
        lt = np.any(F < F[i], axis=1)
        if np.any(le & lt):
            keep[i] = False
    return keep


def unique_rows(F, decimals=12):
    F = np.atleast_2d(np.asarray(F, dtype=float))
    if len(F) == 0:
        return F
    _, idx = np.unique(np.round(F, decimals), axis=0, return_index=True)
    return F[np.sort(idx)]


def extract_front(F):
    """
    Applied the same way to all four. MOEA/D returns its whole population incl.
    dominated members (30 vs NSGA-II's 6), without this it gets scored on junk.
    """
    F = np.atleast_2d(np.asarray(F, dtype=float))
    if len(F) == 0:
        return F
    return unique_rows(F[non_dominated_mask(F)])


# indicators
def hypervolume(F_norm, ref=REF_POINT):
    """Exact 2-D hypervolume by sweep. F_norm: (n,2) minimization, in [0,1]^2."""
    F = np.atleast_2d(np.asarray(F_norm, dtype=float))
    F = F[np.all(F <= ref, axis=1)]           # drop points outside the ref cuboid
    if len(F) == 0:
        return 0.0
    F = unique_rows(F[non_dominated_mask(F)])
    F = F[np.argsort(F[:, 0])]                # ascending f1
    hv, prev_f2 = 0.0, ref[1]
    for f1, f2 in F:
        if f2 < prev_f2:
            hv += (ref[0] - f1) * (prev_f2 - f2)
            prev_f2 = f2
    return float(hv)


def igd(S_norm, P_norm):
    """IGD: mean distance from each true front point to S."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    d = np.linalg.norm(P[:, None, :] - S[None, :, :], axis=2)
    return float(d.min(axis=1).mean())


def igd_plus(S_norm, P_norm):
    """IGD+ (Ishibuchi et al., 2015) -- weakly Pareto-compliant, unlike IGD."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    diff = np.maximum(S[None, :, :] - P[:, None, :], 0.0)   # (|P|, |S|, 2)
    d = np.sqrt((diff ** 2).sum(axis=2))
    return float(d.min(axis=1).mean())


def gd(S_norm, P_norm):
    """GD: mean distance from each point of S to the true front."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    d = np.linalg.norm(S[:, None, :] - P[None, :, :], axis=2)
    return float(d.min(axis=1).mean())


def spacing(S_norm):
    """Schott (1995). Undefined for |S| < 2 -> NaN (frequency of this is reported)."""
    S = np.atleast_2d(S_norm)
    n = len(S)
    if n < 2:
        return float('nan')
    d = np.abs(S[:, None, :] - S[None, :, :]).sum(axis=2)   # L1
    np.fill_diagonal(d, np.inf)
    di = d.min(axis=1)
    return float(np.sqrt(((di.mean() - di) ** 2).sum() / (n - 1)))


def set_coverage(A, B):
    """C(A,B) = fraction of B weakly dominated by A (Zitzler & Thiele 1998). Not symmetric, report both ways."""
    A = np.atleast_2d(A); B = np.atleast_2d(B)
    if len(B) == 0:
        return float('nan')
    if len(A) == 0:
        return 0.0
    covered = 0
    for b in B:
        le = np.all(A <= b, axis=1)
        lt = np.any(A < b, axis=1)
        if np.any(le & lt) or np.any(np.all(A == b, axis=1)):
            covered += 1
    return covered / len(B)


# FE counter, this is what enforces the budget (D-3)
class EvalLog:
    """
    Logs every new candidate evaluated, in order. 1 FE = 1 new candidate.
    Re-evaluating an unchanged individual doesn't count: pymoo never does it, and
    cell 20 recomputes ~3,100 times per run just for logging.
    """

    def __init__(self, enum, budget=FE_BUDGET):
        self.enum = enum
        self.budget = budget
        self.rows = []        # index into enum.F_native, one per new candidate

    def log(self, solution):
        self.rows.append(self.enum.row_of(solution))
        return self.enum.F_native[self.rows[-1]]

    def log_many(self, X):
        for x in X:
            self.log(x)

    @property
    def n_eval(self):
        return len(self.rows)

    def cumulative_front_min(self):
        """Non-dominated subset of the first `budget` evals, so parity is exact."""
        rows = self.rows[:self.budget]
        if not rows:
            return np.empty((0, 2))
        return extract_front(self.enum.F_min[np.array(rows)])


# shared init (D-1), same P0 for all four
def make_initial_population(cache, seed, pop_size=POP_SIZE, target_length=K_SUMMARY):
    """
    cell 20's initialize_population(), but seeded. All four get this P0 for a given
    (article, seed): no init head start, and the paired design makes Wilcoxon signed-rank legit.
    """
    rng = np.random.default_rng(seed)
    n = cache.n
    init_scores = cache.node_importance * cache.position_bonus
    s = init_scores.sum()
    probs = init_scores / s if s > 0 else np.ones(n) / n

    pop = []
    for _ in range(pop_size):
        sol = np.zeros(n, dtype=int)
        num_to_select = max(1, rng.integers(max(1, target_length - 1), target_length + 2))
        chosen = rng.choice(n, size=min(int(num_to_select), n), replace=False, p=probs)
        sol[chosen] = 1
        pop.append(repair_solution(sol, cache, target_length))
    return np.array(pop, dtype=int)


# MO-WOA, cell 20 logic rewired to the shared layer
def _crowding_distance(objectives):
    """Copied from cell 20's crowding_distance()."""
    objectives = np.atleast_2d(objectives)
    n_sol, n_obj = objectives.shape
    if n_sol <= 2:
        return np.full(n_sol, np.inf)
    dist = np.zeros(n_sol)
    for m in range(n_obj):
        order = np.argsort(objectives[:, m])
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng_m = objectives[order[-1], m] - objectives[order[0], m]
        if rng_m == 0:
            continue
        for i in range(1, n_sol - 1):
            dist[order[i]] += (objectives[order[i + 1], m] -
                               objectives[order[i - 1], m]) / rng_m
    return dist


def _pareto_indices(objectives):
    """cell 20's compute_pareto_front in native form (coverage max, redundancy min)."""
    objectives = np.atleast_2d(objectives)
    idx = []
    for i, o1 in enumerate(objectives):
        dominated = False
        for j, o2 in enumerate(objectives):
            if i != j and (o2[0] >= o1[0] and o2[1] <= o1[1] and
                           (o2[0] > o1[0] or o2[1] < o1[1])):
                dominated = True
                break
        if not dominated:
            idx.append(i)
    return idx


def _sigmoid(x):
    """cell 20's sigmoid incl. the [-500, 500] clip. MOPSO uses it too."""
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))


def run_mowoa(enum, seed, pop_size=POP_SIZE, max_iter=MAX_ITER, budget=FE_BUDGET):
    """
    MO-WOA as in cell 20 of [v1_4_1]: binary WOA update, CD leader on the front,
    70/30 leader following, leader kept (so 29 new per gen), repair after every
    update, knee with [32, 1]. FEs: 30 + 50 x 29 = 1480 exactly.
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    population = make_initial_population(cache, seed, pop_size)
    log.log_many(population)                                   # 30 FEs

    gen_history = []          # per-generation population objectives (minimization)

    def objs(pop):            # lookups, not counted as FEs (see EvalLog)
        return np.array([enum.F_of(s) for s in pop])

    def update_position_woa(current, guide, a, pop):
        new_pos = np.zeros(n, dtype=int)
        p = rng.random()
        b = 1.0
        A_coeff = 2 * a * rng.random() - a
        C_coeff = 2 * rng.random()

        if p < 0.5:
            if abs(A_coeff) < 1:                                # shrinking encircling
                ref = guide
            else:                                               # random search
                ref = pop[rng.integers(len(pop))]
            for i in range(n):
                D = abs(C_coeff * ref[i] - current[i])
                val = ref[i] - A_coeff * D
                new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
        else:                                                   # spiral bubble-net
            l = rng.uniform(-1, 1)
            for i in range(n):
                Dp = abs(guide[i] - current[i])
                val = Dp * np.exp(b * l) * np.cos(2 * np.pi * l) + guide[i]
                new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0

        if not new_pos.any():
            new_pos[rng.integers(n)] = 1
        return repair_solution(new_pos, cache)

    for it in range(max_iter):
        a = 2.0 - it * (2.0 / max_iter)
        O = objs(population)
        gen_history.append(np.column_stack([-O[:, 0], O[:, 1]]))

        pidx = _pareto_indices(O)
        if len(pidx) == 0:
            pidx = list(range(len(population)))

        if len(pidx) > 1:
            cd = _crowding_distance(O[pidx])
            inf_mask = np.isinf(cd)
            if np.any(inf_mask):
                lead_in_pareto = rng.choice(np.where(inf_mask)[0])
            else:
                lead_in_pareto = int(np.argmax(cd))
            leader_idx = pidx[lead_in_pareto]
        else:
            leader_idx = pidx[0]
        leader = population[leader_idx]

        new_pop = []
        for i, sol in enumerate(population):
            if i == leader_idx:
                new_pop.append(sol)                             # leader kept
            else:
                guide = leader if rng.random() < 0.7 else population[rng.choice(pidx)]
                child = update_position_woa(sol, guide, a, population)
                log.log(child)                                  # 29 FEs per generation
                new_pop.append(child)
        population = np.array(new_pop)

    O = objs(population)
    gen_history.append(np.column_stack([-O[:, 0], O[:, 1]]))
    F_min_terminal = np.column_stack([-O[:, 0], O[:, 1]])

    # knee point, same as cell 20
    pidx = _pareto_indices(O)
    knee_row = None
    if len(pidx) > 0:
        P = O[pidx]
        omin, omax = P.min(axis=0), P.max(axis=0)
        rngv = omax - omin
        rngv[rngv == 0] = 1
        norm = (P - omin) / rngv
        norm[:, 0] = 1 - norm[:, 0]
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        knee_row = enum.row_of(population[pidx[int(d.argmin())]])

    return dict(
        name='MO-WOA',
        terminal_front=extract_front(F_min_terminal),
        cumulative_front=log.cumulative_front_min(),
        gen_history=gen_history,
        n_eval=log.n_eval,
        knee_row=knee_row,
        eval_rows=log.rows,
    )


# binary MOPSO-CD, hand-coded (pymoo's MOPSO is real-valued only)
def run_mopso(enum, seed, pop_size=POP_SIZE, budget=FE_BUDGET,
              w=0.729, c1=1.49445, c2=1.49445, v_max=6.0, archive_size=POP_SIZE):
    """
    Binary MOPSO-CD (Raquel & Naval, 2005). Used instead of the grid MOPSO
    (Coello Coello, Pulido & Lechuga, 2004) because it picks leaders by crowding
    distance like MO-WOA, so the update equation is the only thing that differs:
        particle : w*v + c1*r1*(pbest - x) + c2*r2*(gbest - x)
        whale    : shrinking encircling + spiral bubble-net + random search
    Same sigmoid, repair, init and budget. Clerc constriction defaults, not tuned (D-4).
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    B = make_initial_population(cache, seed, pop_size)          # same P0 as MO-WOA (D-1)
    log.log_many(B)                                            # 30 FEs

    # continuous position seeded from the binary start (sigmoid(+/-2) ~ 0.88 / 0.12)
    X = np.where(B == 1, 2.0, -2.0).astype(float)
    V = np.zeros((pop_size, n))

    F = np.array([enum.F_of(b) for b in B])                    # native (cov, red)
    F_min = np.column_stack([-F[:, 0], F[:, 1]])

    pbest_B, pbest_Fmin = B.copy(), F_min.copy()
    arch_B, arch_Fmin = _archive_update(B, F_min, np.empty((0, n), int),
                                        np.empty((0, 2)), archive_size)
    gen_history = [F_min.copy()]

    while log.n_eval < budget:
        # leader: binary tournament on crowding distance
        cd = _crowding_distance(arch_Fmin) if len(arch_Fmin) > 1 else np.array([np.inf])
        for i in range(pop_size):
            if len(arch_B) == 1:
                g = arch_B[0]
            else:
                a1, a2 = rng.integers(len(arch_B), size=2)
                g = arch_B[a1] if cd[a1] >= cd[a2] else arch_B[a2]

            r1, r2 = rng.random(n), rng.random(n)
            V[i] = (w * V[i]
                    + c1 * r1 * (np.where(pbest_B[i] == 1, 2.0, -2.0) - X[i])
                    + c2 * r2 * (np.where(g == 1, 2.0, -2.0) - X[i]))
            V[i] = np.clip(V[i], -v_max, v_max)
            X[i] = X[i] + V[i]
            b = (rng.random(n) < _sigmoid(X[i])).astype(int)   # same sigmoid as MO-WOA
            B[i] = repair_solution(b, cache)                   # same repair as MO-WOA

        F = np.array([enum.F_of(b) for b in B])
        F_min = np.column_stack([-F[:, 0], F[:, 1]])
        log.log_many(B)                                        # 30 new candidates
        gen_history.append(F_min.copy())

        # pbest by dominance, tie -> keep new with prob 0.5
        for i in range(pop_size):
            new, old = F_min[i], pbest_Fmin[i]
            if np.all(new <= old) and np.any(new < old):
                pbest_B[i], pbest_Fmin[i] = B[i].copy(), new
            elif not (np.all(old <= new) and np.any(old < new)):
                if rng.random() < 0.5:
                    pbest_B[i], pbest_Fmin[i] = B[i].copy(), new

        arch_B, arch_Fmin = _archive_update(B, F_min, arch_B, arch_Fmin, archive_size)

    return dict(
        name='MOPSO',
        terminal_front=extract_front(arch_Fmin),               # native output is the archive
        cumulative_front=log.cumulative_front_min(),
        gen_history=gen_history,
        n_eval=log.n_eval,
        knee_row=None,
        eval_rows=log.rows,
    )


def _archive_update(B, F_min, arch_B, arch_Fmin, cap):
    """Merge population into the external archive; prune by crowding distance."""
    allB = np.vstack([arch_B, B]) if len(arch_B) else B.copy()
    allF = np.vstack([arch_Fmin, F_min]) if len(arch_Fmin) else F_min.copy()

    keep = non_dominated_mask(allF)
    allB, allF = allB[keep], allF[keep]

    _, uidx = np.unique(np.round(allF, 12), axis=0, return_index=True)
    allB, allF = allB[np.sort(uidx)], allF[np.sort(uidx)]

    if len(allF) > cap:                                        # prune the most crowded
        cd = _crowding_distance(allF)
        keep_idx = np.argsort(-cd)[:cap]
        allB, allF = allB[keep_idx], allF[keep_idx]
    return allB, allF


# pymoo adapters for NSGA-II and MOEA/D
def _pymoo_imports():
    from pymoo.core.problem import Problem
    from pymoo.core.repair import Repair
    from pymoo.core.sampling import Sampling
    from pymoo.core.callback import Callback
    return Problem, Repair, Sampling, Callback


def make_pymoo_components(enum, seed, budget=FE_BUDGET, pop_size=POP_SIZE):
    Problem, Repair, Sampling, Callback = _pymoo_imports()
    cache = enum.cache
    log = EvalLog(enum, budget)

    class SummaryProblem(Problem):
        def __init__(self):
            # needs vtype=bool
            super().__init__(n_var=cache.n, n_obj=2, xl=0, xu=1, vtype=bool)

        def _evaluate(self, X, out, **kw):
            Xi = np.asarray(X).astype(int)
            F = np.array([enum.F_of(x) for x in Xi])
            log.log_many(Xi)                                   # every row = 1 new FE
            out["F"] = np.column_stack([-F[:, 0], F[:, 1]])    # minimization form

    class SummaryRepair(Repair):
        def _do(self, problem, X, **kw):
            # has to return bool. with int64, BitflipMutation flips via bitwise NOT
            # (~0 = -1, ~1 = -2), the encoding gets corrupted and the run still
            # finishes without error (cardinality sums go negative). hit this in testing
            return np.array([repair_solution(x, cache) for x in np.asarray(X).astype(int)]).astype(bool)

    class SharedSampling(Sampling):
        """D-1: every algorithm gets MO-WOA's importance-biased P0."""
        def _do(self, problem, n_samples, **kw):
            P0 = make_initial_population(cache, seed, pop_size)
            if n_samples <= len(P0):
                return P0[:n_samples].astype(bool)
            extra = make_initial_population(cache, seed + 100000,
                                            n_samples - len(P0))
            return np.vstack([P0, extra]).astype(bool)

    class GenCallback(Callback):
        def __init__(self):
            super().__init__()
            self.gen_history = []
        def notify(self, algorithm):
            self.gen_history.append(algorithm.pop.get("F").copy())

    return SummaryProblem(), SummaryRepair(), SharedSampling(), GenCallback(), log


def run_pymoo(enum, seed, which, budget=FE_BUDGET, pop_size=POP_SIZE):
    """
    which: 'NSGA-II' | 'MOEA/D'
    pymoo only checks termination at generation boundaries, so it can't stop at
    exactly 1480 (NSGA-II with ('n_evals', 1500) used 1524). Peers overshoot to
    the next boundary, so the surplus goes to them, never MO-WOA. The cumulative
    front is cut at the first 1480 evals, so parity is exact there.
    """
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.algorithms.moo.moead import MOEAD
    from pymoo.operators.crossover.pntx import TwoPointCrossover
    from pymoo.operators.mutation.bitflip import BitflipMutation
    from pymoo.util.ref_dirs import get_reference_directions
    from pymoo.termination.max_eval import MaximumFunctionCallTermination
    from pymoo.optimize import minimize

    problem, repair, sampling, cb, log = make_pymoo_components(enum, seed, budget, pop_size)

    common = dict(sampling=sampling, crossover=TwoPointCrossover(),
                  mutation=BitflipMutation(), repair=repair)

    if which == 'NSGA-II':
        algo = NSGA2(pop_size=pop_size, eliminate_duplicates=True, **common)
    elif which == 'MOEA/D':
        ref_dirs = get_reference_directions("uniform", 2, n_partitions=pop_size - 1)
        algo = MOEAD(ref_dirs=ref_dirs, n_neighbors=15, **common)
    else:
        raise ValueError(which)

    res = minimize(problem, algo, MaximumFunctionCallTermination(budget),
                   seed=seed, callback=cb, verbose=False, save_history=False)

    # MOEA/D returns its whole population, extract_front cleans that up (same for all four)
    F_terminal = np.atleast_2d(res.pop.get("F"))

    return dict(
        name=which,
        terminal_front=extract_front(F_terminal),
        cumulative_front=log.cumulative_front_min(),
        gen_history=cb.gen_history,
        n_eval=log.n_eval,
        knee_row=None,
        eval_rows=log.rows,
    )


# one (article, seed) -> all four algorithms
ALGORITHMS = ['MO-WOA', 'NSGA-II', 'MOPSO', 'MOEA/D']


def run_all(enum, seed):
    return {
        'MO-WOA':  run_mowoa(enum, seed),
        'NSGA-II': run_pymoo(enum, seed, 'NSGA-II'),
        'MOPSO':   run_mopso(enum, seed),
        'MOEA/D':  run_pymoo(enum, seed, 'MOEA/D'),
    }


def score_run(result, enum, tier='terminal'):
    """Compute every indicator for one run. tier: 'terminal' | 'cumulative' (D-2)."""
    F = result[f'{tier}_front']
    S = enum.normalize(F) if len(F) else np.empty((0, 2))
    P = enum.P_star_norm

    hv = hypervolume(S)
    out = {
        'algorithm': result['name'],
        'tier': tier,
        'HV': hv,
        'HV_pct': 100.0 * hv / enum.hv_star if enum.hv_star > 0 else float('nan'),
        'IGD': igd(S, P),
        'IGD+': igd_plus(S, P),
        'GD': gd(S, P),
        'Spacing': spacing(S),
        'cardinality': len(S),
        'n_eval': result['n_eval'],
    }
    # only possible because we know the true front
    if len(F):
        pf = unique_rows(enum.P_star_min)
        hits = sum(1 for f in F if np.any(np.all(np.isclose(pf, f), axis=1)))
        out['front_recovery'] = hits / len(pf)
        cov_ext = enum.F_min[enum.true_cov_extreme_idx]
        out['found_cov_extreme'] = bool(np.any(np.all(np.isclose(F, cov_ext), axis=1)))
    else:
        out['front_recovery'] = 0.0
        out['found_cov_extreme'] = False
    return out


def hv_convergence(result, enum):
    """HV-vs-generation, normalized as a % of the true optimal HV."""
    return [100.0 * hypervolume(enum.normalize(extract_front(Fg))) / enum.hv_star
            if enum.hv_star > 0 else float('nan')
            for Fg in result['gen_history']]


# MO-WOA+, the fixed variant. canonical MO-WOA above stays untouched.
# pilot diagnosis (real data):
#   retention: terminal 75.70% vs cumulative 87.28% HV, 11.58 pts thrown away.
#     no archive, generational replacement drops what it found
#   search: S-transfer saturates. as a->0, A->0, val -> leader[i] in {0,1},
#     sigmoid(val) -> {0.50, 0.73}, so every bit is a near coin-flip and
#     repair_solution() collapses the offspring onto greedy top-3. diversity dies,
#     search only adds ~4.6 pts in 1480 FEs
# fixes, each switchable for the ablation:
#   use_archive: external archive, cap 30 like MOPSO, non-dominated, crowding-pruned
#     [Coello Coello et al., 2004]
#   transfer='V': V-shaped transfer [Mirjalili & Lewis, 2013; binary WOA: Hussien
#     et al., 2019]. step magnitude -> flip prob of the parent's bit, step->0 keeps the parent

def _v_transfer(step):
    """V-shaped transfer: |tanh(step)|. V(0)=0 (keep parent bit), V(large)->1."""
    return np.abs(np.tanh(step))


def _nds_crowding_select(B_all, F_all, k):
    """(mu+lambda) environmental selection: non-dominated sorting + crowding (NSGA-II)."""
    Nn = len(F_all)
    remaining = np.arange(Nn)
    chosen = []
    while len(chosen) < k and len(remaining) > 0:
        mask = non_dominated_mask(F_all[remaining])
        front = remaining[mask]
        if len(chosen) + len(front) <= k:
            chosen.extend(front.tolist())
            remaining = remaining[~mask]
        else:
            cd = _crowding_distance(F_all[front])
            need = k - len(chosen)
            chosen.extend(front[np.argsort(-cd)[:need]].tolist())
            break
    chosen = np.array(chosen[:k])
    return B_all[chosen], F_all[chosen]


def run_mowoa_final(enum, seed, transfer='V', guide_mode='per_whale_archive',
                    survival='elitist', use_archive=True,
                    ls_probes=0, ls_mode='directed',
                    pop_size=POP_SIZE, max_iter=MAX_ITER, budget=FE_BUDGET,
                    archive_size=POP_SIZE):
    """
    Ablation-switchable MO-WOA+. Update equations, repair, init (D-1), budget (D-3)
    and knee weights are the same as cell 20. Switches:
      transfer   : 'S' (cell 20 sigmoid) | 'V' (V-shaped flip)
      guide_mode : 'single_leader' (cell 20) | 'per_whale_archive' (MOPSO-CD gbest rule)
      survival   : 'generational' (cell 20) | 'elitist' (mu+lambda NDS+crowding, like NSWOA)
      use_archive: external archive, cap 30, crowding-pruned
    S / single_leader / generational / no archive reproduces run_mowoa() bit-exactly (G7b).
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    population = make_initial_population(cache, seed, pop_size)
    log.log_many(population)

    def objs(pop):
        return np.array([enum.F_of(s) for s in pop])

    F = objs(population)
    F_min = np.column_stack([-F[:, 0], F[:, 1]])
    gen_history = [F_min.copy()]

    arch_B = np.empty((0, n), dtype=int); arch_F = np.empty((0, 2))
    if use_archive:
        arch_B, arch_F = _archive_update(population, F_min, arch_B, arch_F, archive_size)

    def offspring_of(current, guide, a, pop):
        p = rng.random(); b = 1.0
        A_coeff = 2 * a * rng.random() - a
        C_coeff = 2 * rng.random()
        if transfer == 'S':
            new_pos = np.zeros(n, dtype=int)
            if p < 0.5:
                ref = guide if abs(A_coeff) < 1 else pop[rng.integers(len(pop))]
                for i in range(n):
                    D = abs(C_coeff * ref[i] - current[i])
                    val = ref[i] - A_coeff * D
                    new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
            else:
                l = rng.uniform(-1, 1)
                for i in range(n):
                    Dp = abs(guide[i] - current[i])
                    val = Dp * np.exp(b * l) * np.cos(2 * np.pi * l) + guide[i]
                    new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
        else:
            new_pos = current.copy()
            if p < 0.5:
                ref = guide if abs(A_coeff) < 1 else pop[rng.integers(len(pop))]
                for i in range(n):
                    D = abs(C_coeff * ref[i] - current[i])
                    if rng.random() < _v_transfer(A_coeff * D):
                        new_pos[i] = 1 - new_pos[i]
            else:
                l = rng.uniform(-1, 1)
                for i in range(n):
                    Dp = abs(guide[i] - current[i])
                    if rng.random() < _v_transfer(Dp * np.exp(b * l) * np.cos(2 * np.pi * l)):
                        new_pos[i] = 1 - new_pos[i]
        if not new_pos.any():
            new_pos[rng.integers(n)] = 1
        return repair_solution(new_pos, cache)

    def ls_probe():
        """1-swap probe on an archive member. Cardinality stays at k so repair never
        fires, it's the only move that skips repair's importance bias.
        random: uniform swap. directed: drop the selected sentence most similar to
        the others, add one of the 3 least similar unselected ones."""
        if len(arch_B) == 0:
            return None
        if len(arch_B) == 1:
            base = arch_B[0]
        else:
            cdp = _crowding_distance(arch_F)
            a1, a2 = rng.integers(len(arch_B), size=2)
            base = arch_B[a1] if cdp[a1] >= cdp[a2] else arch_B[a2]
        x = np.asarray(base).astype(int).copy()
        sel = np.where(x == 1)[0]
        unsel = np.where(x == 0)[0]
        if len(sel) == 0 or len(unsel) == 0:
            return None
        if ls_mode == 'random':
            out_i = rng.choice(sel)
            in_i = rng.choice(unsel)
        else:
            contrib = np.array([cache.adj[i, sel].sum() for i in sel])
            out_i = sel[int(np.argmax(contrib))] if rng.random() < 0.7 else rng.choice(sel)
            remaining = sel[sel != out_i]
            scores = cache.adj[unsel][:, remaining].sum(axis=1)
            k = min(3, len(unsel))
            in_i = unsel[rng.choice(np.argsort(scores)[:k])]
        x[out_i] = 0
        x[in_i] = 1
        return x

    for it in range(max_iter):
        a = 2.0 - it * (2.0 / max_iter)
        F = objs(population)
        F_min = np.column_stack([-F[:, 0], F[:, 1]])
        pidx = _pareto_indices(F)
        if len(pidx) == 0:
            pidx = list(range(len(population)))

        # leader / guides
        if guide_mode == 'single_leader':
            if len(pidx) > 1:
                cd = _crowding_distance(F[pidx])
                inf_mask = np.isinf(cd)
                lp = (rng.choice(np.where(inf_mask)[0]) if np.any(inf_mask)
                      else int(np.argmax(cd)))
                leader_idx = pidx[lp]
            else:
                leader_idx = pidx[0]
            leader = population[leader_idx]
        else:
            leader_idx = 0
            leader = None

        def draw_guide():
            if guide_mode == 'per_whale_archive' and use_archive and len(arch_B) > 0:
                if len(arch_B) == 1:
                    return arch_B[0]
                cd = _crowding_distance(arch_F)
                a1, a2 = rng.integers(len(arch_B), size=2)
                return arch_B[a1] if cd[a1] >= cd[a2] else arch_B[a2]
            if guide_mode == 'per_whale_archive':          # archive off: pop front
                if len(pidx) == 1:
                    return population[pidx[0]]
                cd = _crowding_distance(F[pidx])
                a1, a2 = rng.integers(len(pidx), size=2)
                return population[pidx[a1] if cd[a1] >= cd[a2] else pidx[a2]]
            return leader

        # 29 offspring, leader stays in its slot like cell 20 (needed for the
        # bit-exact rng stream at baseline switches)
        new_pop = []
        children = []
        if ls_probes > 0 and use_archive and len(arch_B) > 0:
            nonleader = [i for i in range(len(population)) if i != leader_idx]
            probe_slots = set(rng.choice(nonleader,
                                         size=min(ls_probes, len(nonleader)),
                                         replace=False).tolist())
        else:
            probe_slots = set()
        for i, sol in enumerate(population):
            if i == leader_idx:
                new_pop.append(sol)
                continue
            child = None
            if i in probe_slots:
                child = ls_probe()
            if child is None:
                g = draw_guide() if rng.random() < 0.7 else population[rng.choice(pidx)]
                child = offspring_of(sol, g, a, population)
            log.log(child)
            new_pop.append(child)
            children.append(child)
        children = np.array(children)
        Fc = objs(children)
        Fc_min = np.column_stack([-Fc[:, 0], Fc[:, 1]])

        # survival
        if survival == 'generational':
            population = np.array(new_pop)
        else:                                              # elitist (mu+lambda)
            B_all = np.vstack([population, children])
            F_all = np.vstack([F_min, Fc_min])
            population, _ = _nds_crowding_select(B_all, F_all, pop_size)

        Fq = objs(population)
        Fq_min = np.column_stack([-Fq[:, 0], Fq[:, 1]])
        gen_history.append(Fq_min.copy())
        if use_archive:
            arch_B, arch_F = _archive_update(children, Fc_min, arch_B, arch_F,
                                             archive_size)

    # terminal output + knee
    if use_archive:
        F_terminal = arch_F; knee_B, knee_F = arch_B, arch_F
    else:
        Fq = objs(population)
        F_terminal = np.column_stack([-Fq[:, 0], Fq[:, 1]])
        pidx = _pareto_indices(Fq)
        knee_B, knee_F = population[pidx], F_terminal[pidx]

    knee_row = None
    if len(knee_F) > 0:
        native = np.column_stack([-knee_F[:, 0], knee_F[:, 1]])
        omin, omax = native.min(axis=0), native.max(axis=0)
        rngv = omax - omin; rngv[rngv == 0] = 1
        norm = (native - omin) / rngv
        norm[:, 0] = 1 - norm[:, 0]
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        knee_row = enum.row_of(knee_B[int(d.argmin())])

    return dict(name=f"MO-WOA+[{transfer},{guide_mode[:8]},{survival[:4]}"
                     f"{',arch' if use_archive else ''}]",
                terminal_front=extract_front(F_terminal),
                cumulative_front=log.cumulative_front_min(),
                gen_history=gen_history, n_eval=log.n_eval,
                knee_row=knee_row, eval_rows=log.rows)

## C2 - load data and the phase C records

In [ ]:
import importlib, mowoa_mo_shared as mo
importlib.reload(mo)
import pymoo

LOG = []
def out(*a):
    s = ' '.join(str(x) for x in a)
    print(s)
    LOG.append(s)

with open(CACHE_PATH, 'rb') as f:
    caches = [mo.ArticleCache.from_dict(d) for d in pickle.load(f)]
enums = {c.article_id: mo.Enumeration(c) for c in caches}
AIDS = sorted(enums)
HARD = [a for a in AIDS if enums[a].M > mo.FE_BUDGET]
EASY = [a for a in AIDS if enums[a].M <= mo.FE_BUDGET]
GROUP = {a: ('hard' if a in HARD else 'easy') for a in AIDS}
M_OF = {a: enums[a].M for a in AIDS}

out('=' * 78)
out('WP4 / C3 run —', time.strftime('%Y-%m-%d %H:%M'), '| python', platform.python_version(),
    '| numpy', np.__version__, '| pymoo', pymoo.__version__, '| CPUs', os.cpu_count(), '| workers', WORKERS)
out(f'articles {len(AIDS)} (hard {len(HARD)}, easy {len(EASY)}) | seeds {SEEDS[0]}..{SEEDS[-1]} ({len(SEEDS)})')

SCAL = ['article', 'algorithm', 'seed', 'n_eval', 'term_HV_pct', 'cumu_HV_pct', 'term_front_recovery',
        'cumu_front_recovery', 'term_found_cov_extreme', 'term_cardinality']
rows = []
for a in AIDS:
    with open(f'{PHASEC_DIR}/article_{a:03d}.pkl', 'rb') as f:
        for r in pickle.load(f):
            if r['seed'] in SEEDS:
                rows.append({k: r[k] for k in SCAL})
PC = pd.DataFrame(rows)
PC['group'] = PC.article.map(GROUP)
ALGOS = ['MO-WOA', 'MO-WOA+', 'NSGA-II', 'MOPSO', 'MOEA/D']
assert set(PC.algorithm) == set(ALGOS), sorted(set(PC.algorithm))
assert len(PC) == len(AIDS) * len(SEEDS) * len(ALGOS), 'Phase C incomplete for these seeds'
out(f'Phase C runs loaded: {len(PC)} ({len(ALGOS)} algorithms x {len(AIDS)} articles x {len(SEEDS)} seeds)')

BASELINE = dict(transfer='S', guide_mode='single_leader', survival='generational', use_archive=False)
FROZEN = dict(transfer='V', guide_mode='per_whale_archive', survival='elitist', use_archive=True,
              ls_probes=10, ls_mode='directed')
PEERS = {'NSGA-II': lambda e, s: mo.run_pymoo(e, s, 'NSGA-II'),
         'MOPSO':   lambda e, s: mo.run_mopso(e, s),
         'MOEA/D':  lambda e, s: mo.run_pymoo(e, s, 'MOEA/D')}


## C3 - Q17: comparator settings as this pymoo version resolves them

In [ ]:
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.termination.max_eval import MaximumFunctionCallTermination

def val(x):
    return getattr(x, 'value', x)

def cname(x):
    return type(x).__name__

e0 = enums[HARD[0]]
SETTINGS = {}
for which in ('NSGA-II', 'MOEA/D'):
    prob_, rep_, samp_, cb_, _ = mo.make_pymoo_components(e0, 0)
    common = dict(sampling=samp_, crossover=TwoPointCrossover(), mutation=BitflipMutation(), repair=rep_)
    if which == 'NSGA-II':           # built the same way as in run_pymoo
        algo = NSGA2(pop_size=mo.POP_SIZE, eliminate_duplicates=True, **common)
    else:
        algo = MOEAD(ref_dirs=get_reference_directions('uniform', 2, n_partitions=mo.POP_SIZE - 1),
                     n_neighbors=15, **common)
    algo.setup(prob_, termination=MaximumFunctionCallTermination(mo.FE_BUDGET), seed=0)
    cx, mu = algo.mating.crossover, algo.mating.mutation
    d = dict(pop_size=algo.pop_size, n_offsprings=getattr(algo, 'n_offsprings', None),
             crossover=f'{cname(cx)} (n_points={getattr(cx, "n_points", "?")}, prob={val(cx.prob)})',
             mutation=(f'{cname(mu)} (prob={val(mu.prob)}, prob_var='
                       f'{"None -> min(0.5, 1/N) per article" if mu.prob_var is None else val(mu.prob_var)}; '
                       f'N = {e0.cache.n} here -> {float(mu.get_prob_var(prob_)):.4f})'),
             duplicate_elimination=cname(algo.eliminate_duplicates),
             repair='shared repair_solution (cardinality 3)', sampling='shared P0 (D-1)')
    if which == 'NSGA-II':
        sel = algo.mating.selection
        d['selection'] = (f'{cname(sel)} (pressure={getattr(sel, "pressure", "?")}, '
                          f'{getattr(getattr(sel, "func_comp", None), "__name__", "?")})')
        d['survival'] = f'{cname(algo.survival)} (crowding: {cname(getattr(algo.survival, "crowding_func", None))})'
    else:
        d['weight_vectors'] = f'{len(algo.ref_dirs)} uniform (n_partitions={mo.POP_SIZE - 1})'
        d['n_neighbors'] = algo.n_neighbors
        d['decomposition'] = cname(algo.decomposition)
        d['prob_neighbor_mating'] = val(getattr(algo.selection, 'prob', None))
    SETTINGS[which] = d
_sig = inspect.signature(mo.run_mopso).parameters
SETTINGS['MOPSO'] = dict(
    implementation='hand-coded binary MOPSO-CD (shared module, numpy only)',
    inertia_w=_sig['w'].default, c1=_sig['c1'].default, c2=_sig['c2'].default, v_max=_sig['v_max'].default,
    archive=f'{mo.POP_SIZE} (non-dominated, crowding-pruned)', leader='binary tournament on crowding distance',
    pbest='Pareto dominance; mutual non-dominance -> new with p 0.5',
    transfer='sigmoid (S-shaped), bit = 1 if U < sigmoid(x)', repair='shared repair_solution', sampling='shared P0 (D-1)')
out('\n[C3] Q17 — COMPARATOR SETTINGS (pymoo ' + pymoo.__version__ + ')')
for which, d in SETTINGS.items():
    out(f'  {which}')
    for k, v_ in d.items():
        out(f'    {k:<22} {v_}')
out('  No comparator uses the V-shaped transfer, the external archive of MO-WOA+ or local-search probes.')


## C4 - Q1 and Q3 from the phase C records (no reruns)

In [ ]:
def block(g):
    rec_t, rec_c = g.term_front_recovery.to_numpy(float), g.cumu_front_recovery.to_numpy(float)
    ex_t, ex_c = np.isclose(rec_t, 1.0), np.isclose(rec_c, 1.0)
    return dict(runs=len(g),
                eval_mean=g.n_eval.mean(), eval_min=int(g.n_eval.min()), eval_max=int(g.n_eval.max()),
                eval_eq_1480_pct=100 * (g.n_eval == mo.FE_BUDGET).mean(),
                eval_below_1480_runs=int((g.n_eval < mo.FE_BUDGET).sum()),
                hv_term_mean=g.term_HV_pct.mean(), hv_term_min=g.term_HV_pct.min(),
                rec_term_mean=100 * rec_t.mean(), rec_term_min=100 * rec_t.min(),
                exact_term_pct=100 * ex_t.mean(),
                hv_cumu_mean=g.cumu_HV_pct.mean(), rec_cumu_mean=100 * rec_c.mean(),
                exact_cumu_pct=100 * ex_c.mean(),
                lost_after_visit_pct=100 * (~ex_t & ex_c).mean(),      # visited all of P*, not all kept
                never_visited_pct=100 * (~ex_t & ~ex_c).mean(),        # some P* member never generated
                cov_extreme_pct=100 * g.term_found_cov_extreme.astype(bool).mean())

Q13 = []
for algo in ALGOS:
    for grp in ('hard', 'easy', 'all'):
        g = PC[PC.algorithm == algo] if grp == 'all' else PC[(PC.algorithm == algo) & (PC.group == grp)]
        Q13.append(dict(algorithm=algo, group=grp, **block(g)))
Q13 = pd.DataFrame(Q13)

out('\n[C4] Q1 — EVALUATIONS CONSUMED PER RUN (Phase C, all articles)')
out(f"  {'algorithm':<9} {'mean':>8} {'min':>6} {'max':>6} {'=1480 %':>8} {'runs <1480':>11}")
for r in Q13[Q13.group == 'all'].itertuples():
    out(f'  {r.algorithm:<9} {r.eval_mean:>8.1f} {r.eval_min:>6} {r.eval_max:>6} '
        f'{r.eval_eq_1480_pct:>8.1f} {r.eval_below_1480_runs:>11}')
for grp in ('hard', 'easy'):
    out(f'  {grp}: ' + ', '.join(f'{r.algorithm} {r.eval_mean:.1f} [{r.eval_min}, {r.eval_max}]'
                                 for r in Q13[Q13.group == grp].itertuples()))

out(f'\n[C4] Q3 — FRONT RECOVERY ON THE {len(EASY)} EASY ARTICLES (C(N,3) <= {mo.FE_BUDGET}), '
    f'{len(EASY) * len(SEEDS)} runs per algorithm')
out(f"  {'algorithm':<9} {'HV mean':>8} {'HV min':>7} {'rec mean':>9} {'rec min':>8} {'exact %':>8} "
    f"{'visited-lost %':>15} {'never-visited %':>16} {'cumu exact %':>13} {'cov-ext %':>10}")
for r in Q13[Q13.group == 'easy'].itertuples():
    out(f'  {r.algorithm:<9} {r.hv_term_mean:>8.2f} {r.hv_term_min:>7.2f} {r.rec_term_mean:>9.1f} '
        f'{r.rec_term_min:>8.1f} {r.exact_term_pct:>8.1f} {r.lost_after_visit_pct:>15.1f} '
        f'{r.never_visited_pct:>16.1f} {r.exact_cumu_pct:>13.1f} {r.cov_extreme_pct:>10.1f}')
out('  exact = terminal output contains every true-front point; visited-lost = every true-front point was '
    'generated within 1,480 evaluations but not all were kept; never-visited = at least one was never generated.')
E_ = PC[PC.group == 'easy'].assign(ex=lambda d: np.isclose(d.term_front_recovery.astype(float), 1.0))
per_art = E_.groupby(['algorithm', 'article']).ex.agg(['all', 'any', 'mean']).reset_index()
out(f'  easy articles with exact recovery in all {len(SEEDS)} seeds / in no seed:')
for algo in ALGOS:
    p = per_art[per_art.algorithm == algo]
    out(f'    {algo:<9} all seeds {int(p["all"].sum()):>3}/{len(EASY)} | no seed {int((~p["any"]).sum()):>3}/{len(EASY)}')
out('\n  hard articles (for reference): ' + ', '.join(
    f'{r.algorithm} HV {r.hv_term_mean:.2f}, recovery {r.rec_term_mean:.1f}%, exact {r.exact_term_pct:.1f}%'
    for r in Q13[Q13.group == 'hard'].itertuples()))


## C5 - Q2: rerun the 3 comparators and keep every evaluated candidate

In [ ]:
def run_parallel(fn, jobs):
    if WORKERS <= 1:
        for j in jobs:
            yield fn(*j)
        return
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=get_context('fork')) as ex:
        futs = [ex.submit(fn, *j) for j in jobs]
        for f in as_completed(futs):
            yield f.result()

def work_article(aid, todo):
    e = enums[aid]
    recs = []
    for algo, seed in todo:
        t0 = time.perf_counter()
        with contextlib.redirect_stdout(io.StringIO()):            # pymoo mating warnings
            r = PEERS[algo](e, seed)
        sec = time.perf_counter() - t0
        rows_ = list(r['eval_rows'])
        st, sc = mo.score_run(r, e, 'terminal'), mo.score_run(r, e, 'cumulative')
        recs.append(dict(article=aid, algorithm=algo, seed=seed, sec=sec, n_eval=r['n_eval'],
                         n_unique=len(set(rows_)), n_unique_1480=len(set(rows_[:mo.FE_BUDGET])),
                         hv=st['HV_pct'], recov=st['front_recovery'], cum_recov=sc['front_recovery']))
    return aid, recs

def load_ck(aid):
    ck = f'{RUN_DIR}/article_{aid:03d}.pkl'
    if not os.path.exists(ck):
        return []
    with open(ck, 'rb') as f:
        return pickle.load(f)

# quick check before the long rerun: a few hard-article runs have to match phase C exactly
_pk = PC.set_index(['article', 'algorithm', 'seed'])
_bad = []
for aid in HARD[:3]:
    for algo, seed in (('NSGA-II', SEEDS[0]), ('MOEA/D', SEEDS[0]), ('MOPSO', SEEDS[0])):
        r = work_article(aid, [(algo, seed)])[1][0]
        ref = _pk.loc[(aid, algo, seed)]
        if r['n_eval'] != ref.n_eval or not np.isclose(r['hv'], ref.term_HV_pct):
            _bad.append((aid, algo, seed, r['n_eval'], int(ref.n_eval)))
if _bad:
    raise RuntimeError(f'pymoo {pymoo.__version__} does not reproduce Phase C on {_bad} '
                       '(article, algorithm, seed, n_eval now, n_eval Phase C). Stop here.')
print('pre-check passed: rerun reproduces Phase C on 3 hard articles x 3 comparators')

jobs = []
for aid in AIDS:
    have = {(r['algorithm'], r['seed']) for r in load_ck(aid)}
    todo = [(p, sd) for p in PEERS for sd in SEEDS if (p, sd) not in have]
    if todo:
        jobs.append((aid, todo))
jobs.sort(key=lambda j: -enums[j[0]].M * len(j[1]))
n_runs = sum(len(t) for _, t in jobs)
print(f'comparator reruns: {n_runs} runs to do over {len(jobs)} articles with {WORKERS} worker(s)')
t0, done = time.time(), 0
for aid, recs in run_parallel(work_article, jobs):
    old = load_ck(aid)
    with open(f'{RUN_DIR}/article_{aid:03d}.pkl', 'wb') as f:      # written by the parent
        pickle.dump(old + recs, f)
    done += len(recs)
    el = time.time() - t0
    print(f'  article {aid:>3} saved | {done}/{n_runs} runs | {el / 60:.1f} min | '
          f'ETA {el / done * (n_runs - done) / 60:.0f} min')

PR = pd.DataFrame([r for aid in AIDS for r in load_ck(aid)])
PR = PR[PR.seed.isin(SEEDS)].reset_index(drop=True)
assert len(PR) == len(AIDS) * len(PEERS) * len(SEEDS), 'missing runs — rerun this cell'
out(f'\n[C5] comparator reruns: {len(PR)} ({len(PEERS)} algorithms x {len(AIDS)} articles x {len(SEEDS)} seeds)')


## C6 - reproduction checks against phase C

In [ ]:
out('\nREPRODUCTION CHECKS (rerun vs Phase C, same article and seed)')
mg = PR.merge(PC[['article', 'algorithm', 'seed', 'n_eval', 'term_HV_pct', 'term_front_recovery',
                  'cumu_front_recovery']], on=['article', 'algorithm', 'seed'], suffixes=('', '_pc'))
assert len(mg) == len(PR)
REPRO = {}
for algo in PEERS:
    g = mg[mg.algorithm == algo]
    ok_ne = (g.n_eval == g.n_eval_pc).sum()
    ok_hv = np.isclose(g.hv, g.term_HV_pct).sum()
    ok_rc = (np.isclose(g.recov, g.term_front_recovery) & np.isclose(g.cum_recov, g.cumu_front_recovery)).sum()
    REPRO[algo] = (ok_ne == len(g)) and (ok_hv == len(g)) and (ok_rc == len(g))
    out(f'  {algo:<8} n_eval {ok_ne}/{len(g)} | terminal HV {ok_hv}/{len(g)} | recovery (both tiers) {ok_rc}/{len(g)}'
        f'  -> {"REPRODUCED" if REPRO[algo] else "NOT REPRODUCED"}')
if not all(REPRO.values()):
    out(f'  !! pymoo {pymoo.__version__} does not reproduce Phase C for '
        f'{[a for a, ok in REPRO.items() if not ok]}. The unique counts below then describe this version, '
        f'not the reported runs. Check this before using them.')


## C7 - results: attempted vs unique evaluations (Q2), space visited (Q3)

In [ ]:
W2 = pd.read_csv(WP2_RUNS)
W2['article'] = W2.article - 1                                   # WP2 CSV is 1-based
W2 = W2[W2.seed.isin(SEEDS)]
STAGE_ALGO = {'Baseline': 'MO-WOA', '+A': 'MO-WOA +A', '+A+E': 'MO-WOA +A+E', '+A+E+G': 'MO-WOA +A+E+G',
              '+A+E+G+V': 'MO-WOA +A+E+G+V', '+LS': 'MO-WOA+'}
assert set(W2.stage) == set(STAGE_ALGO) and len(W2) == len(AIDS) * len(STAGE_ALGO) * len(SEEDS), \
    'wp2_runs.csv incomplete for these seeds'
W2 = W2.assign(algorithm=W2.stage.map(STAGE_ALGO), n_eval=mo.FE_BUDGET, n_unique_1480=W2.n_unique)
out(f'\n  check: MO-WOA+ unique candidates per run (WP2 runs) mean {W2[W2.algorithm == "MO-WOA+"].n_unique.mean():.1f} '
    f'(WP1: 88.3)')
COLS = ['article', 'algorithm', 'seed', 'n_eval', 'n_unique', 'n_unique_1480']
U = pd.concat([W2[COLS], PR[COLS]], ignore_index=True)
U['M'] = U.article.map(M_OF)
U['group'] = U.article.map(GROUP)
U['unique_pct'] = 100 * U.n_unique / U.n_eval
U['space_pct'] = 100 * U.n_unique / U.M
U['visited_all'] = U.n_unique == U.M
ORDER = ['MO-WOA', 'MO-WOA +A', 'MO-WOA +A+E', 'MO-WOA +A+E+G', 'MO-WOA +A+E+G+V', 'MO-WOA+', 'NSGA-II', 'MOPSO', 'MOEA/D']

TAB = []
for algo in ORDER:
    for grp in ('hard', 'easy', 'all'):
        g = U[U.algorithm == algo] if grp == 'all' else U[(U.algorithm == algo) & (U.group == grp)]
        TAB.append(dict(algorithm=algo, group=grp, runs=len(g),
                        attempted_mean=g.n_eval.mean(), attempted_min=int(g.n_eval.min()), attempted_max=int(g.n_eval.max()),
                        unique_mean=g.n_unique.mean(), unique_median=g.n_unique.median(),
                        unique_min=int(g.n_unique.min()), unique_max=int(g.n_unique.max()),
                        unique_in_first_1480_mean=g.n_unique_1480.mean(),
                        repeats_mean=(g.n_eval - g.n_unique).mean(), unique_pct_mean=g.unique_pct.mean(),
                        space_visited_pct_mean=g.space_pct.mean(), visited_entire_space_pct=100 * g.visited_all.mean()))
TAB = pd.DataFrame(TAB)
for grp, label in (('hard', f'{len(HARD)} HARD articles'), ('easy', f'{len(EASY)} EASY articles')):
    out(f'\n[C7] Q2 — ATTEMPTED vs UNIQUE EVALUATIONS PER RUN, {label} (mean [min, max])')
    out(f"  {'algorithm':<17} {'attempted':>20} {'unique':>18} {'unique %':>9} {'repeats':>8} "
        f"{'% of C(N,3) visited':>20} {'whole space %':>14}")
    for r in TAB[TAB.group == grp].itertuples():
        out(f'  {r.algorithm:<17} {r.attempted_mean:>7.1f} [{r.attempted_min:>4}, {r.attempted_max:>4}] '
            f'{r.unique_mean:>6.1f} [{r.unique_min:>4}, {r.unique_max:>4}] {r.unique_pct_mean:>9.1f} '
            f'{r.repeats_mean:>8.1f} {r.space_visited_pct_mean:>20.1f} {r.visited_entire_space_pct:>14.1f}')
out('  attempted = candidates generated and counted against the budget (repeats included); '
    'unique = distinct 3-sentence subsets among them; no algorithm caches evaluations '
    '(every generated candidate is a lookup in the same enumerated table and is counted once per generation).')

# Q3 link: on easy articles, exact recovery vs share of the feasible set visited
ex = PC[['article', 'algorithm', 'seed', 'term_front_recovery']].copy()
ex['exact'] = np.isclose(ex.term_front_recovery.astype(float), 1.0)
L = U.merge(ex, on=['article', 'algorithm', 'seed'], how='inner')
L = L[L.group == 'easy']
out(f'\n[C7] Q3 — EASY ARTICLES: exact terminal recovery by share of the feasible set visited')
out(f"  {'algorithm':<9} {'space visited %':>16} {'exact | whole space':>20} {'exact | not whole':>18}")
for algo in ALGOS:
    g = L[L.algorithm == algo]
    w, nw = g[g.visited_all], g[~g.visited_all]
    fmt = lambda s: f'{100 * s.exact.mean():.1f}% (n={len(s)})' if len(s) else 'n=0'
    out(f'  {algo:<9} {g.space_pct.mean():>16.1f} {fmt(w):>20} {fmt(nw):>18}')

# R5.8: where the MO-WOA vs comparator gap comes from (terminal HV%, hard articles,
# seed mean per article, paired across articles, descriptive 95% t-interval, no test)
HVA = {STAGE_ALGO[s]: W2[(W2.stage == s) & W2.article.isin(HARD)].groupby('article').hv.mean()
       for s in ('Baseline', '+A+E+G', '+A+E+G+V', '+LS')}
HVA.update({a: PC[(PC.algorithm == a) & (PC.group == 'hard')].groupby('article').term_HV_pct.mean()
            for a in ('NSGA-II', 'MOPSO', 'MOEA/D')})
out(f'\n  check: MO-WOA+ hard HV from WP2 runs {HVA["MO-WOA+"].mean():.2f} vs Phase C '
    f'{PC[(PC.algorithm == "MO-WOA+") & (PC.group == "hard")].term_HV_pct.mean():.2f}')
def paired(a, b):
    d = (HVA[a] - HVA[b]).dropna()
    m, se = d.mean(), d.std(ddof=1) / np.sqrt(len(d))
    h = stats.t.ppf(0.975, len(d) - 1) * se
    return m, m - h, m + h, int((d > 1e-9).sum()), int((d.abs() <= 1e-9).sum()), int((d < -1e-9).sum()), len(d)
PAIRS = [('MO-WOA +A+E+G', 'MOPSO'), ('MO-WOA +A+E+G+V', 'MOPSO'), ('MO-WOA+', 'MOPSO'),
         ('MO-WOA', 'MOPSO'), ('MO-WOA+', 'NSGA-II'), ('MO-WOA+', 'MOEA/D')]
out(f'\n[C7] R5.8 — TERMINAL HV%, {len(HARD)} HARD ARTICLES: paired article-level differences '
    '(seed mean per article; descriptive 95% t-interval; higher/equal/lower articles)')
out('  means: ' + ', '.join(f'{k} {v.mean():.2f}' for k, v in HVA.items()))
PAIRED = []
for a, b in PAIRS:
    m, lo, hi, w, t_, l_, n = paired(a, b)
    PAIRED.append(dict(a=a, b=b, diff=m, lo=lo, hi=hi, higher=w, equal=t_, lower=l_, n=n))
    out(f'  {a:<17} - {b:<8} {m:+7.2f} [{lo:+.2f}, {hi:+.2f}]  {w}/{t_}/{l_} (n={n})')
out('  MOPSO-CD shares with MO-WOA +A+E+G: S-shaped transfer, external archive of 30 with crowding pruning, '
    'crowding-tournament guide/leader, shared repair and P0. It has no V-shaped transfer and no local-search probes.')


## C8 - optional: sequential runtime on the 33 hard articles (Q20 / R1.3 data)

In [ ]:
TIM = pd.DataFrame()
if RUN_TIMING:
    TREAT = {'MO-WOA': lambda e, s: mo.run_mowoa_final(e, s, **BASELINE),
             'MO-WOA+': lambda e, s: mo.run_mowoa_final(e, s, **FROZEN), **PEERS}
    cpu = ''
    try:
        with open('/proc/cpuinfo') as f:
            cpu = next((l.split(':', 1)[1].strip() for l in f if l.startswith('model name')), '')
    except OSError:
        pass
    with contextlib.redirect_stdout(io.StringIO()):
        for name, fn in TREAT.items():                               # warm-up (imports, caches), not timed
            fn(enums[HARD[0]], 0)
    tim = []
    t_all = time.time()
    for a in HARD:
        for s in TIMING_SEEDS:
            for name, fn in TREAT.items():
                t0 = time.perf_counter()
                with contextlib.redirect_stdout(io.StringIO()):
                    fn(enums[a], s)
                tim.append(dict(article=a, algorithm=name, seed=s, sec=time.perf_counter() - t0))
    TIM = pd.DataFrame(tim)
    out(f'\n[C8] RUNTIME — one process, {len(HARD)} hard articles x seeds {TIMING_SEEDS}, '
        f'{time.time() - t_all:.0f} s in total')
    out(f'  hardware: {cpu or platform.processor()} | {os.cpu_count()} logical CPUs | python {platform.python_version()} '
        f'| numpy {np.__version__} | pymoo {pymoo.__version__}')
    out('  objective evaluation = lookup in the pre-computed enumeration (BGE-M3 embedding and graph building excluded)')
    out(f"  {'algorithm':<9} {'median s/run':>13} {'mean s/run':>11} {'max s/run':>10}")
    for name in TREAT:
        g = TIM[TIM.algorithm == name].sec
        out(f'  {name:<9} {g.median():>13.3f} {g.mean():>11.3f} {g.max():>10.3f}')


## C9 - save

In [ ]:
PA = TAB.merge(Q13, on=['algorithm', 'group'], how='left', suffixes=('', '_phaseC'))
PA.round(4).to_csv(f'{WP4_DIR}/wp4_per_algorithm.csv', index=False)
pd.DataFrame(PAIRED).round(4).to_csv(f'{WP4_DIR}/wp4_paired_hv.csv', index=False)
PR.assign(article=PR.article + 1).round(5).to_csv(f'{WP4_DIR}/wp4_peer_runs.csv', index=False)
if len(TIM):
    TIM.assign(article=TIM.article + 1).round(4).to_csv(f'{WP4_DIR}/wp4_timing.csv', index=False)
with open(f'{WP4_DIR}/wp4_summary.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(LOG) + '\n')
print('\nsaved to', WP4_DIR, ':', sorted(os.listdir(WP4_DIR)))
print('files to keep: wp4_summary.txt, wp4_per_algorithm.csv')
